In [2]:
# =============================================================================
# INDIVIDUAL METHOD RUNNER
# Quick testing of a single method on a single dataset
# =============================================================================

# -----------------------------------------------------------------------------
# CONFIGURATION - CHANGE THESE VALUES
# -----------------------------------------------------------------------------

METHOD = "tabr"           # Method to run (e.g., 'xgboost', 'catboost', 'tabpfn', 'mlp')
DATASET = "0001.gmsc"        # Dataset name (e.g., '0014.hmeq', '0001.gmsc')
TASK = "pd"                  # Task type: 'pd' (classification) or 'lgd' (regression)

# -----------------------------------------------------------------------------
# FIXED SETTINGS (for quick testing)
# -----------------------------------------------------------------------------

ROW_LIMIT = 10000             # Limit rows for fast execution
MAX_EPOCHS = 15              # Max epochs for deep learning methods
CV_SPLITS = 1                # Single fold
TUNE = False                 # No HPO
SEED = 42                    # Random seed
TEST_SIZE = 0.2              # Test set fraction
VAL_SIZE = 0.2               # Validation set fraction

# -----------------------------------------------------------------------------
# SETUP
# -----------------------------------------------------------------------------

import sys
from pathlib import Path
import pickle
import json
from datetime import datetime

# Add project root to path (notebook is in notebooks/ folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"\n{'='*60}")
print(f" Running: {METHOD} on {DATASET} ({TASK.upper()})")
print(f"{'='*60}")
print(f"  Row limit:  {ROW_LIMIT}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  CV splits:  {CV_SPLITS}")
print(f"  HPO:        {TUNE}")
print(f"{'='*60}\n")

# -----------------------------------------------------------------------------
# RUN METHOD
# -----------------------------------------------------------------------------

from src.methods.method_runner import run_talent_method, get_available_methods

# Show available methods
available = get_available_methods()
print(f"Available classical methods: {available['classical']}")
print(f"Available deep methods: {available['deep'][:10]}... ({len(available['deep'])} total)")
print()

# Run the method
results = run_talent_method(
    task=TASK,
    dataset=DATASET,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    cv_splits=CV_SPLITS,
    seed=SEED,
    row_limit=ROW_LIMIT,
    method=METHOD,
    max_epoch=MAX_EPOCHS,
    tune=TUNE,
    verbose=True,
)

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------

print(f"\n{'='*60}")
print(f" RESULTS")
print(f"{'='*60}")

for fold_id, fold_results in results.items():
    print(f"\nFold {fold_id}:")
    print(f"  Train time: {fold_results['train_time']:.2f}s")
    print(f"  Samples:    {len(fold_results['y_true'])}")
    
    if TASK == 'lgd':
        print(f"  Clipped:    {fold_results['n_clipped_below']} below, {fold_results['n_clipped_above']} above")
    
    print(f"\n  Metrics:")
    for metric_name, metric_value in fold_results['metrics'].items():
        if not (isinstance(metric_value, float) and metric_value != metric_value):  # Skip NaN
            print(f"    {metric_name:20s}: {metric_value:.4f}")

# -----------------------------------------------------------------------------
# SAVE RESULTS
# -----------------------------------------------------------------------------

# Create output directory
output_dir = PROJECT_ROOT / 'results' / 'individual_method_runner'
output_dir.mkdir(parents=True, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"{METHOD}_{DATASET}_{TASK}_{timestamp}"

# Save as pickle (full results)
pickle_path = output_dir / f"{filename}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\nResults saved to: {pickle_path}")

# Save summary as JSON (metrics only, for easy viewing)
summary = {
    'method': METHOD,
    'dataset': DATASET,
    'task': TASK,
    'timestamp': timestamp,
    'config': {
        'row_limit': ROW_LIMIT,
        'max_epochs': MAX_EPOCHS,
        'cv_splits': CV_SPLITS,
        'tune': TUNE,
        'seed': SEED,
    },
    'folds': {}
}

for fold_id, fold_results in results.items():
    summary['folds'][fold_id] = {
        'train_time': fold_results['train_time'],
        'n_samples': len(fold_results['y_true']),
        'metrics': {k: v for k, v in fold_results['metrics'].items() if not (isinstance(v, float) and v != v)},
    }
    if TASK == 'lgd':
        summary['folds'][fold_id]['n_clipped_below'] = fold_results['n_clipped_below']
        summary['folds'][fold_id]['n_clipped_above'] = fold_results['n_clipped_above']

json_path = output_dir / f"{filename}.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to: {json_path}")

print(f"\n{'='*60}")
print(f" DONE")
print(f"{'='*60}")

Project root: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit

 Running: tabr on 0001.gmsc (PD)
  Row limit:  10000
  Max epochs: 15
  CV splits:  1
  HPO:        False

Available classical methods: ['LinearRegression', 'LogReg', 'NCM', 'NaiveBayes', 'RandomForest', 'catboost', 'dummy', 'knn', 'lightgbm', 'svm', 'xgboost']
Available deep methods: ['amformer', 'autoint', 'bishop', 'danets', 'dcn2', 'dnnr', 'excelformer', 'ftt', 'grande', 'grownet']... (38 total)


Running tabr (deep) on 0001.gmsc (PD)

Preparing data with 1 CV splits...
Fold IDs: [1]
First fold ID: 1

Directory setup:
  Config directory (persistent): C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\config_hpo\pd\0001.gmsc\tabr\NO_HPO
  Checkpoint directory (temp):   C:\Users\U0152019\AppData\Local\Temp\talent_ckpt_0001.gmsc_tabr_nc366b6a

[HPO] Mode: DISABLED
[HPO] All folds: Will use TALENT's default hyperparameters

Fold 1/1
using gpu: 0
{'batch_

2it [00:00,  3.31it/s]


epoch 0, val, loss=0.2342 classification result=0.9363
Epoch: 0, Time cost: 8.716331720352173
epoch 1, train 1/7, loss=0.2153 lr=0.0003121
best epoch 0, best val res=0.9363


2it [00:00,  3.26it/s]


epoch 1, val, loss=0.1973 classification result=0.9306
Epoch: 1, Time cost: 8.781364440917969
epoch 2, train 1/7, loss=0.2006 lr=0.0003121
best epoch 0, best val res=0.9363


...

epoch 13, val, loss=0.1787 classification result=0.9344
Epoch: 13, Time cost: 8.873348712921143
epoch 14, train 1/7, loss=0.1587 lr=0.0003121
best epoch 10, best val res=0.9394


2it [00:00,  2.92it/s]


epoch 14, val, loss=0.1772 classification result=0.9344
Epoch: 14, Time cost: 9.271437883377075
best epoch 10, best val res=0.9394


2it [00:00,  2.54it/s]

Test: loss=0.1892
[Accuracy]=0.9365
[Avg_Recall]=0.5330
[Avg_Precision]=0.7347
[F1]=0.1241
[LogLoss]=0.1892
[AUC]=0.8622

Fold 1 metrics:
  Optimal_Threshold: 0.0100
  AUC: 0.8630
  Gini: 0.7260
  Avg_Precision: 0.2968
  KS: 0.5969
  LogLoss: 1.9706
  Accuracy: 0.9350
  Balanced_Accuracy: 0.5541
  F1: 0.1875
  Precision: 0.4688
  Recall: 0.1172
  MCC: 0.2109

Completed 1 folds for tabr


 RESULTS

Fold 1:
  Train time: 138.97s
  Samples:    2000

  Metrics:
    Optimal_Threshold   : 0.0100
    AUC                 : 0.8630
    Gini                : 0.7260
    Avg_Precision       : 0.2968
    KS                  : 0.5969
    LogLoss             : 1.9706
    Accuracy            : 0.9350
    Balanced_Accuracy   : 0.5541
    F1                  : 0.1875
    Precision           : 0.4688
    Recall              : 0.1172
    MCC                 : 0.2109

Results saved to: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\individual_method_runner\tabr_